In [1]:
from datetime import datetime

from transformers.utils import replace_variables_in_comments

s = '2026-03-18 15:57:19.062'
s = datetime.strptime(s, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%dT%H:%M:%S.%f')[:-3]
print(s)

2026-03-18T15:57:19.062


In [2]:
import yaml
from paths import VARIABLE_CONFIG_PATH

with open(VARIABLE_CONFIG_PATH, 'r', encoding='utf-8') as f:
    variable_mapping = yaml.safe_load(f)

In [13]:
import sqlglot
from sqlglot import exp


def remove_timestamp_from_ast_v2(node: exp.Expression, dialect: str = "hive") -> exp.Expression:
    """Accepts a sqlglot AST node and mutates/returns it with the following

    expressions removed from any underlying SELECT clauses:

    1. Columns with the alias 'etl_timestamp'
    2. Columns selecting the literal/parameter '${batch_timestamp}'
    3. Columns selecting the function CURRENT_TIMESTAMP()
    """

    def transformer(node_to_transform):
        if isinstance(node_to_transform, exp.Select):
            new_select_expressions = []

            for expression in node_to_transform.expressions:
                # Rule 1: Exclude by Alias Name ('etl_timestamp')
                if isinstance(expression, exp.Alias) and expression.alias.lower() == (
                    "etl_timestamp"
                ):
                    continue

                # Extract the inner expression behind the 'AS' alias if it exists
                unaliased_expr = (
                    expression.this if isinstance(expression, exp.Alias) else expression
                )

                # Rule 2: Exclude by Literal/Parameter Value ('${batch_timestamp}')
                raw_expr_string = unaliased_expr.sql(dialect=dialect).strip("'\"")
                if raw_expr_string.lower() == "${batch_timestamp}":
                    continue

                # Rule 3: Exclude by Function Type (CURRENT_TIMESTAMP())
                if isinstance(unaliased_expr, exp.CurrentTimestamp):
                    continue

                # Keep the expression if it passes all filters
                new_select_expressions.append(expression)

            # Re-assign the filtered expressions to the SELECT node
            node_to_transform.set("expressions", new_select_expressions)
            return node_to_transform

        return node_to_transform

    return node.transform(transformer)


# ==========================================
# VERIFICATION
# ==========================================
if __name__ == "__main__":
    dialect = "hive"
    sql_input = """
    INSERT OVERWRITE TABLE ${com_schema}.temp_table
        SELECT
          T0.RECORD_ID,
          CURRENT_TIMESTAMP() AS ETL_TIMESTAMP,  -- Matches Rule 1 & 3
          CURRENT_TIMESTAMP(),                   -- Matches Rule 3 (No Alias)
          CURRENT_TIMESTAMP() AS GENERIC_TIME,   -- Matches Rule 3 (Different Alias)
          '${batch_timestamp}' AS OLD_TIME,      -- Matches Rule 2
          T0.BUSINESS_CODE
        FROM ${com_schema}.temp_t_lmskibb2_tbl_counterparty_all AS T0;
    """

    ast_node = sqlglot.parse_one(sql_input, read=dialect)
    modified_ast = remove_timestamp_from_ast_v2(ast_node, dialect=dialect)

    print(modified_ast.sql(dialect=dialect, pretty=True))

INSERT OVERWRITE TABLE ${com_schema}.temp_table
SELECT
  T0.RECORD_ID,
  T0.BUSINESS_CODE
FROM ${com_schema}.temp_t_lmskibb2_tbl_counterparty_all AS T0


In [3]:
"import sqlglot



# 1. Đọc nội dung tệp SQL
with open(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\logic_cleaner\before.sql", 'r') as file:
    sql_content = file.read()

original_ast = sqlglot.parse(sql_content, read='hive')
original_ast

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\dungp\\projects\\hql_spark_bridge\\docs\\reference\\logic_cleaner\\before.sql'

In [ ]:
from transformers.utils import replace_table_identifier, strip_partition_clauses

# from src.transformers.utils import replace_table_identifier

ast_bien_doi = replace_table_identifier(
    node=original_ast[0],
    old_schema="${com_schema}",
    old_table="t_mhbos_m_client",
    new_schema="${com_schema}",
    new_table="temp_t_mhbos_m_client_consolidated",
    dialect="hive" # Hoặc spark
)

ast_bien_doi = strip_partition_clauses(ast_bien_doi)

# In ra kết quả
print(ast_bien_doi.sql(dialect="hive", pretty=True))

In [ ]:
replace_variabled = replace_variables_in_comments(original_ast[0], variable_mapping)


with open(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\logic_cleaner\function_test\hilarious_comment.sql", "w") as f:
    f.write(replace_variabled.sql(dialect='hive', pretty=True) + ";")

In [6]:
from migration.cur.key_detector import CurKeyDetector
from utils.file_utils import parse_file_name
from migration.cur.decomposer import CurSqlDecomposer
from paths import *
from src.utils.source_rule_loader import load_all_source_rules

# input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_r_k2_cif_alias.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_m21_a_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_mhbos_m_client.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_account.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_account_bank.sql"
input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_iremisier_account.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()[source_name] if source_name in all_source_rules else all_source_rules['default']

print(f"File gốc tại: {input_file}")

# def run_migration_pipeline():
# ==========================================
# BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
# ==========================================
decomposer = CurSqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

# Ghi file sub-SQL ra ổ đĩa
CurKeyDetector().detect(decomposed_script, source_rules)


File gốc tại: C:\Users\ext_giadung\projects\datalake-script\dml\cur\cur_iremisier_account.sql


{'logical_primary_key': ['ACCOUNT_ID'],
 'confidence': 'MEDIUM',
 'reasoning': 'Columns repeatedly used in >= 2 LEFT JOIN conditions.'}

In [ ]:
ACCOUNT_ID, BANK_ACCOUNT_TYPE, BANK_ACCOUNT_LOCALE

In [ ]:
# src/migration/decomposer.py
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Dict, List, Set
import sqlglot
import sqlglot.expressions as exp


content = input_file.read_text(encoding="utf-8")

header_lines, sql_lines = [], []
is_header = True
for line in content.split('\n'):
    stripped = line.strip()
    if is_header and (stripped.startswith('--') or not stripped):
        header_lines.append(line)
    else:
        is_header = False
        sql_lines.append(line)

header_comments = '\n'.join(header_lines).strip()
sql_content = '\n'.join(sql_lines)


statements = sqlglot.parse(sql_content, read="hive", error_level=sqlglot.ErrorLevel.WARN)
temp_table_registry = defaultdict(set)


In [ ]:
# --- VÒNG 1: Xây dựng Registry (Ai sở hữu bảng nào?) ---
for stmt in statements:
    if stmt is None:
        continue

    source_id = decomposer._strategy_partition_key(stmt)
    print(source_id)


In [ ]:
for a in statements[-1].find_all(exp.Partition):
    print(a)

In [ ]:

{'TOMS': {'TEMP_DIM_ACCOUNT_CONTACT',
  'TEMP_DIM_ACCOUNT_CONTACT_TOMS_ECORPORATE',
  'TEMP_DIM_TRADER_CONTACT'},
 'MHBOS': {'TEMP_DIM_ACCOUNT_CONTACT',
  'TEMP_DIM_BRANCH_CONTACT',
  'TEMP_DIM_TRADER_CONTACT'},
 'GUAVA': {'TEMP_DIM_ACCOUNT_CONTACT', 'TEMP_DIM_ACCOUNT_CONTACT_GUAVA'},
 'M21': {'TEMP_DIM_ACCOUNT_CONTACT',
  'TEMP_DIM_ACCOUNT_CONTACT_M21',
  'TEMP_DIM_TRADER_CONTACT'},
 'KDI': {'TEMP_DIM_ACCOUNT_CONTACT'},
 'RAK': {'TEMP_DIM_ACCOUNT_CONTACT'},
 'SBL': {'TEMP_DIM_ACCOUNT_CONTACT'},
 'LMS': {'TEMP_DIM_ACCOUNT_CONTACT'}}


# test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_lms_test.sql")
# test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_toms_test.sql")
# test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_m21_test.sql")
# test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_m21_create.sql")
test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_create.sql")
# test_file = Path(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\testing\dim_contact_test.sql")

statements = sqlglot.parse(test_file.read_text(encoding="utf-8"), read="hive", error_level=sqlglot.ErrorLevel.WARN)

# _determine_source_recursive(statements[0], 'dim_contact', test_registry)
# {
#     "source_key_strategy": _analyze_statement_source_key_strategy(statements[0], 'dim_contact'),
#     "source_name_strategy": _analyze_statement_source_name_strategy(statements[0], 'dim_contact'),
#     "select_table_strategy": _analyze_statement_select_table_strategy(statements[0], 'dim_contact')
# }

analyze_source_id(statements[0], 'dim_contact', test_registry)

In [ ]:
test = statements[0].find(exp.With).expressions[1].this

isinstance(test, exp.Union)

_find_literal_assignment(test.expression, "SOURCE_NAME")

In [ ]:
get_source_tables(statements[0])


for node in statements[0].find_all((exp.Insert, exp.Create, exp.Drop, exp.TruncateTable)):
    if isinstance(node.this, exp.Schema):
        print(node.this.this.name)
    if isinstance(node.this, exp.Table):
        print(table.name)

In [ ]:
statements[0].this.this.name

In [ ]:
for table in statements[0].expression.find_all(exp.Table):
    print(table.name)

In [ ]:
schema = KeyDetectorV2()._get_schema(decomposed_script, source_rules)
schema

In [ ]:
from src.migration.ddl_resolver import DdlResolver
from src.migration.schema_extractor import SchemaExtractor

ddl_resolver = DdlResolver(source_rules = source_rules)
ddl_path = ddl_resolver.resolve_ddl_path(input_file)

schema_extractor = SchemaExtractor(source_rules)
columns = schema_extractor.extract(ddl_path)

[column['name'] for column in columns]